# SHROOM DeBERTa LoRA Validation Set Fine-Tuning Runner - Colab / VS Code

In [ ]:
# 1) Configuration
from pathlib import Path
from datetime import datetime

# Google Drive project folder containing finetune_deberta_lora_v2.py, src/, data/, participant_kit/.
DRIVE_PROJECT = Path("/content/drive/MyDrive/thesis_colab/model_experiments_colab")

# Temporary Colab workspace used for actual execution.
LOCAL_PROJECT = Path("/content/model_experiments_colab")

# Where this runner backs up outputs and logs in Google Drive.
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/thesis_colab/outputs_deberta_lora_finetune_colab_vscode")

# One timestamped backup folder for this session.
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_RUN_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_TAG

# Fine-tuning script expected in DRIVE_PROJECT / LOCAL_PROJECT.
SCRIPT_NAME = "finetune_deberta_lora_v2.py"

# Dataset paths relative to LOCAL_PROJECT.
TRAIN_PATH = "data/SHROOM_dev-v2/val.model-agnostic.json"
EVAL_SPLIT = 0.2
WARMUP_PATH = "data/SHROOM_trial-v1.1/trial-v1.json"
WARMUP_N = 10

# DeBERTa LoRA validation recipe selected from local validation pilots.
SEED = 42
PREVIEW_N = 2
LABEL_MODE = "soft"
EPOCHS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.1
MAX_LENGTH = 512
BEST_METRIC = "rho"

# Precision.
FP16 = False
BF16 = False

# Optional attention control.
ATTN_IMPLEMENTATION = None 

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "query_proj,key_proj,value_proj"
LORA_MODULES_TO_SAVE = "classifier,pooler"
LORA_BIAS = "none"

# Optional smoke-test limits. Leave as None for full internal split runs.
TRAIN_LIMIT = None
EVAL_LIMIT = None

# Participant scorer controls.
# Keep both False for internal dev split runs.
WRITE_CURRENT = False
RUN_PARTICIPANT_SCORER = False

# Execution controls.
CONTINUE_ON_ERROR = True
SKIP_EXISTING = True 
DRY_RUN = False        

# Optional filters.
# Examples:
# ONLY_CONTAINS = "xsmall"       # only cross-encoder xsmall
# ONLY_CONTAINS = "sileod"       # only supplementary tasksource model
# ONLY_CONTAINS = ""             # all enabled jobs below
ONLY_CONTAINS = ""
ONLY_MODEL_NAMES = []  # Optional exact model-name list. Leave empty to ignore.

# If True, remove LOCAL_PROJECT/outputs after syncing from Drive.
CLEAR_LOCAL_OUTPUTS = False

print("Drive project:", DRIVE_PROJECT)
print("Local project:", LOCAL_PROJECT)
print("Drive output folder for this run:", DRIVE_RUN_OUTPUT_DIR)
print("Run tag:", RUN_TAG)
print("Script:", SCRIPT_NAME)

Drive project: /content/drive/MyDrive/thesis_colab/model_experiments_colab
Local project: /content/model_experiments_colab
Drive output folder for this run: /content/drive/MyDrive/thesis_colab/outputs_deberta_lora_finetune_colab_vscode/20260510_111929
Run tag: 20260510_111929
Script: finetune_deberta_lora_v2.py


In [ ]:
# 2) Install/check dependencies and GPU status
import subprocess
import sys
import platform

INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    packages = [
        "transformers<5",
        "accelerate",
        "peft",
        "sentencepiece",
        "protobuf<6",
        "scipy",
        "scikit-learn",
        "huggingface_hub",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("gpu_total_memory_gb:", round(props.total_memory / 1024**3, 2))
        print("bf16_supported:", torch.cuda.is_bf16_supported())
except Exception as exc:
    print("Could not inspect torch/GPU:", repr(exc))

try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception as exc:
    print("nvidia-smi unavailable:", repr(exc))

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.10.0+cu128
cuda_available: True
gpu_name: Tesla T4
gpu_total_memory_gb: 14.56
bf16_supported: True


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'torchao'], returncode=0)

In [ ]:
# 3) Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Could not import/mount google.colab.drive. Are you connected to a Colab runtime?")
    raise

Mounted at /content/drive


In [ ]:
# 4) Sync project from Google Drive to /content
import os
import shutil
from pathlib import Path

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Drive project folder not found: {DRIVE_PROJECT}\n"
        "Create/upload your folder at this path, or edit DRIVE_PROJECT in cell 1."
    )

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

ignore = shutil.ignore_patterns(
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ipynb_checkpoints",
    "wandb",
)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, ignore=ignore)
os.chdir(LOCAL_PROJECT)

if CLEAR_LOCAL_OUTPUTS:
    outputs_dir = LOCAL_PROJECT / "outputs"
    if outputs_dir.exists():
        shutil.rmtree(outputs_dir)

print("Working directory:", Path.cwd())
print("Top-level files:")
for p in sorted(Path.cwd().iterdir()):
    print("-", p.name)

Working directory: /content/model_experiments_colab
Top-level files:
- data
- finetune_deberta.py
- finetune_deberta_final_eval.py
- finetune_deberta_lora_v2.py
- finetune_flan_lora.py
- finetune_flan_lora_final_eval.py
- finetune_gemma_lora_final_eval.py
- finetune_gemma_lora_v2.py
- finetune_qwen_lora.py
- finetune_qwen_lora_final_eval.py
- participant_kit
- run_experiment.py
- run_experiment_ootb_eval.py
- src


In [ ]:
# 5) Sanity-check expected files before running
from pathlib import Path

required_paths = [
    SCRIPT_NAME,
    "src/data.py",
    "data/SHROOM_dev-v2/val.model-agnostic.json",
    "data/SHROOM_trial-v1.1/trial-v1.json",
]

if RUN_PARTICIPANT_SCORER:
    required_paths.extend([
        "participant_kit/check_output.py",
        "participant_kit/score.py",
    ])

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print("-", p)
    raise FileNotFoundError("Project structure check failed.")

print("Project structure looks good.")

Project structure looks good.


In [ ]:
# 6) Fine-tuning job list: DeBERTa LoRA size rungs and supplementary NLI variants
FINETUNE_JOBS = [
    {
        "model_name": "cross-encoder/nli-deberta-v3-xsmall",
        "run": False,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "cross-encoder/nli-deberta-v3-small",
        "run": False,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "cross-encoder/nli-deberta-v3-base",
        "run": False,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "cross-encoder/nli-deberta-v3-large",
        "run": False,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
        "run": True,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "sileod/deberta-v3-base-tasksource-nli",
        "run": True,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
]

print("Configured DeBERTa LoRA fine-tuning jobs:")
for job in FINETUNE_JOBS:
    eff_bs = job["train_batch_size"] * job["grad_accum_steps"]
    print(
        f"- {job['model_name']} | run={job['run']} | "
        f"train_bs={job['train_batch_size']} | eval_bs={job['eval_batch_size']} | "
        f"grad_accum={job['grad_accum_steps']} | effective_bs={eff_bs} | "
        f"grad_ckpt={job['gradient_checkpointing']}"
    )

Configured DeBERTa LoRA fine-tuning jobs:
- cross-encoder/nli-deberta-v3-xsmall | run=False | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- cross-encoder/nli-deberta-v3-small | run=False | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- cross-encoder/nli-deberta-v3-base | run=False | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- cross-encoder/nli-deberta-v3-large | run=False | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli | run=True | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- sileod/deberta-v3-base-tasksource-nli | run=True | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False


In [ ]:
# 7) Optional Hugging Face login
LOGIN_TO_HF = False

if LOGIN_TO_HF:
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token is None:
        import getpass
        token = getpass.getpass("Paste Hugging Face token: ")

    login(token=token)
    print("Logged into Hugging Face.")
else:
    print("HF login skipped.")

HF login skipped.


In [ ]:
# 8) Runner helpers: filtering, streamed subprocess logs, existing-run checks, and Drive backup
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def safe_filename(text: str) -> str:
    return (
        text.replace("/", "__")
        .replace("\\", "__")
        .replace(":", "_")
        .replace(" ", "_")
    )

def selected_jobs(jobs: list[dict]) -> list[dict]:
    only_contains = ONLY_CONTAINS.strip().lower()
    exact = set(ONLY_MODEL_NAMES or [])

    selected = []
    for item in jobs:
        if not item.get("run", True):
            continue
        if exact and item["model_name"] not in exact:
            continue
        if only_contains and only_contains not in item["model_name"].lower():
            continue
        selected.append(item)
    return selected

def existing_metadata_for_model(model_name: str) -> list[Path]:
    safe_model = safe_filename(model_name)
    metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
    if not metadata_dir.exists():
        return []
    return sorted(metadata_dir.glob(f"run__deberta_lora__finetuned__{LABEL_MODE}__{safe_model}__*.json"))

def backup_outputs_to_drive() -> None:
    DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    src_outputs = LOCAL_PROJECT / "outputs"
    dst_outputs = DRIVE_RUN_OUTPUT_DIR / "outputs"

    if src_outputs.exists():
        if dst_outputs.exists():
            shutil.rmtree(dst_outputs)
        shutil.copytree(src_outputs, dst_outputs)

    print(f"Backed up outputs -> {dst_outputs}")

def stream_subprocess(command: list[str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)

    print("Command:", " ".join(command))
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

        process.wait()
        return int(process.returncode)

def job_flag(job: dict, key: str, default):
    return job[key] if key in job else default

def build_command(job: dict) -> list[str]:
    command = [
        sys.executable,
        SCRIPT_NAME,
        "--model-name", job["model_name"],
        "--train-path", TRAIN_PATH,
        "--eval-split", str(EVAL_SPLIT),
        "--seed", str(SEED),
        "--max-length", str(MAX_LENGTH),
        "--epochs", str(EPOCHS),
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--warmup-ratio", str(WARMUP_RATIO),
        "--train-batch-size", str(job["train_batch_size"]),
        "--eval-batch-size", str(job["eval_batch_size"]),
        "--grad-accum-steps", str(job["grad_accum_steps"]),
        "--label-mode", LABEL_MODE,
        "--best-metric", BEST_METRIC,
        "--preview-n", str(PREVIEW_N),
        "--warmup-path", WARMUP_PATH,
        "--warmup-n", str(WARMUP_N),
        "--run-tag", RUN_TAG,
        "--notes", f"Colab VS Code DeBERTa LoRA internal split; aligned final LoRA recipe; run_tag={RUN_TAG}",
        "--lora-r", str(LORA_R),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
        "--lora-target-modules", LORA_TARGET_MODULES,
        "--lora-modules-to-save", LORA_MODULES_TO_SAVE,
        "--lora-bias", LORA_BIAS,
    ]

    if job_flag(job, "fp16", FP16):
        command.append("--fp16")
    if job_flag(job, "bf16", BF16):
        command.append("--bf16")
    if job.get("gradient_checkpointing", False):
        command.append("--gradient-checkpointing")
    if ATTN_IMPLEMENTATION:
        command.extend(["--attn-implementation", str(ATTN_IMPLEMENTATION)])
    if TRAIN_LIMIT is not None:
        command.extend(["--train-limit", str(TRAIN_LIMIT)])
    if EVAL_LIMIT is not None:
        command.extend(["--eval-limit", str(EVAL_LIMIT)])
    if WRITE_CURRENT:
        command.append("--write-current")
    if RUN_PARTICIPANT_SCORER:
        command.append("--run-participant-scorer")

    return command

def run_one_job(job: dict) -> dict:
    model_name = job["model_name"]
    safe_model = safe_filename(model_name)
    log_path = DRIVE_RUN_OUTPUT_DIR / "logs" / f"{safe_model}.log"

    print("\n" + "=" * 100)
    print(f"Running fine-tune: {model_name}")
    print("=" * 100)

    existing = existing_metadata_for_model(model_name)
    if SKIP_EXISTING and existing:
        print(f"SKIP_EXISTING=True and found existing metadata for {model_name}:")
        for path in existing[-3:]:
            print("-", path)
        return {
            "model_name": model_name,
            "status": "skipped_existing",
            "returncode": None,
            "started_at": datetime.now().isoformat(timespec="seconds"),
            "ended_at": datetime.now().isoformat(timespec="seconds"),
            "log_path": None,
        }

    command = build_command(job)
    started_at = datetime.now().isoformat(timespec="seconds")

    if DRY_RUN:
        print("DRY RUN; not executing.")
        print("Command:", " ".join(command))
        return {
            "model_name": model_name,
            "status": "dry_run",
            "returncode": 0,
            "started_at": started_at,
            "ended_at": datetime.now().isoformat(timespec="seconds"),
            "log_path": str(log_path),
        }

    try:
        returncode = stream_subprocess(command, log_path)
    finally:
        backup_outputs_to_drive()

    ended_at = datetime.now().isoformat(timespec="seconds")
    status = "success" if returncode == 0 else "failed"
    print(f"{status.upper()}: {model_name} with return code {returncode}")

    return {
        "model_name": model_name,
        "status": status,
        "returncode": returncode,
        "started_at": started_at,
        "ended_at": ended_at,
        "log_path": str(log_path),
    }

In [ ]:
# 9) Optional smoke/dry run
print("DRY_RUN:", DRY_RUN)
print("ONLY_CONTAINS:", ONLY_CONTAINS)
print("ONLY_MODEL_NAMES:", ONLY_MODEL_NAMES)
print("TRAIN_PATH:", TRAIN_PATH)
print("EVAL_SPLIT:", EVAL_SPLIT)
print("LABEL_MODE:", LABEL_MODE)
print("EPOCHS:", EPOCHS)
print("LEARNING_RATE:", LEARNING_RATE)
print("BEST_METRIC:", BEST_METRIC)
print("FP16:", FP16)
print("BF16:", BF16)
print("ATTN_IMPLEMENTATION:", ATTN_IMPLEMENTATION)
print("LoRA:", {
    "r": LORA_R,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "targets": LORA_TARGET_MODULES,
    "modules_to_save": LORA_MODULES_TO_SAVE,
})

selected = selected_jobs(FINETUNE_JOBS)
print(f"Selected {len(selected)} job(s):")
for job in selected:
    print(f"- {job['model_name']}")
    print("  ", " ".join(build_command(job)))

DRY_RUN: False
ONLY_CONTAINS: 
ONLY_MODEL_NAMES: []
TRAIN_PATH: data/SHROOM_dev-v2/val.model-agnostic.json
EVAL_SPLIT: 0.2
LABEL_MODE: soft
EPOCHS: 4
LEARNING_RATE: 0.0002
BEST_METRIC: rho
FP16: False
BF16: False
ATTN_IMPLEMENTATION: None
LoRA: {'r': 16, 'alpha': 32, 'dropout': 0.05, 'targets': 'query_proj,key_proj,value_proj', 'modules_to_save': 'classifier,pooler'}
Selected 2 job(s):
- MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
   /usr/bin/python3 finetune_deberta_lora_v2.py --model-name MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli --train-path data/SHROOM_dev-v2/val.model-agnostic.json --eval-split 0.2 --seed 42 --max-length 512 --epochs 4 --learning-rate 0.0002 --weight-decay 0.0 --warmup-ratio 0.1 --train-batch-size 1 --eval-batch-size 2 --grad-accum-steps 4 --label-mode soft --best-metric rho --preview-n 2 --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10 --run-tag 20260510_111929 --notes Colab VS Code DeBERTa LoRA internal split; aligned 

In [ ]:
# 10) Run selected fine-tuning jobs and back up after each one
DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DRIVE_RUN_OUTPUT_DIR / "run_manifest.json"
results = []

selected = selected_jobs(FINETUNE_JOBS)
if not selected:
    raise ValueError("No jobs selected. Check ONLY_CONTAINS / ONLY_MODEL_NAMES / run flags.")

for job in selected:
    result = run_one_job(job)
    results.append(result)

    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "run_tag": RUN_TAG,
                "drive_project": str(DRIVE_PROJECT),
                "local_project": str(LOCAL_PROJECT),
                "drive_run_output_dir": str(DRIVE_RUN_OUTPUT_DIR),
                "script_name": SCRIPT_NAME,
                "train_path": TRAIN_PATH,
                "eval_split": EVAL_SPLIT,
                "label_mode": LABEL_MODE,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
                "best_metric": BEST_METRIC,
                "precision": {
                    "fp16": FP16,
                    "bf16": BF16,
                },
                "lora": {
                    "r": LORA_R,
                    "alpha": LORA_ALPHA,
                    "dropout": LORA_DROPOUT,
                    "target_modules": LORA_TARGET_MODULES,
                    "modules_to_save": LORA_MODULES_TO_SAVE,
                    "bias": LORA_BIAS,
                },
                "warmup_path": WARMUP_PATH,
                "warmup_n": WARMUP_N,
                "seed": SEED,
                "results": results,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    if result["status"] == "failed" and not CONTINUE_ON_ERROR:
        raise RuntimeError(f"Stopping after failed run: {job['model_name']}")

print("\nBatch complete. Manifest:", manifest_path)
print(json.dumps(results, indent=2))


Running fine-tune: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Command: /usr/bin/python3 finetune_deberta_lora_v2.py --model-name MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli --train-path data/SHROOM_dev-v2/val.model-agnostic.json --eval-split 0.2 --seed 42 --max-length 512 --epochs 4 --learning-rate 0.0002 --weight-decay 0.0 --warmup-ratio 0.1 --train-batch-size 1 --eval-batch-size 2 --grad-accum-steps 4 --label-mode soft --best-metric rho --preview-n 2 --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10 --run-tag 20260510_111929 --notes Colab VS Code DeBERTa LoRA internal split; aligned final LoRA recipe; run_tag=20260510_111929 --lora-r 16 --lora-alpha 32 --lora-dropout 0.05 --lora-target-modules query_proj,key_proj,value_proj --lora-modules-to-save classifier,pooler --lora-bias none
2026-05-10 11:35:40.051025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performan

In [ ]:
# 11) Summarize DeBERTa LoRA fine-tuning metadata files found in outputs/metadata
import json
from pathlib import Path

metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
rows = []

if metadata_dir.exists():
    for path in sorted(metadata_dir.glob("run__deberta_lora__finetuned__*.json")):
        try:
            meta = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print("Could not read", path, exc)
            continue

        if meta.get("model_type") != "deberta_lora_finetuned_binary_classifier":
            continue

        comp = meta.get("computational_cost", {})
        hp = meta.get("hyperparameters", {})
        lora = meta.get("lora", {})
        final = meta.get("final_prediction_metrics", {})
        best = meta.get("best_eval_metrics_during_training") or {}

        history = meta.get("training_history", []) or []
        best_epoch = None
        best_rho = best.get("rho")
        if best_rho is not None:
            for item in history:
                ev = item.get("eval") or {}
                if ev.get("rho") == best_rho:
                    best_epoch = item.get("epoch")
                    break

        rows.append({
            "base_model_name": meta.get("base_model_name"),
            "parameter_count": comp.get("parameter_count"),
            "trainable_params": comp.get("trainable_parameter_count"),
            "trainable_frac": comp.get("trainable_parameter_fraction"),
            "label_mode": meta.get("label_mode"),
            "split_mode": meta.get("split_mode"),
            "num_train": meta.get("num_train_examples"),
            "num_eval": meta.get("num_eval_examples"),
            "epochs": hp.get("epochs"),
            "lr": hp.get("learning_rate"),
            "effective_bs": hp.get("effective_train_batch_size"),
            "fp16": hp.get("fp16"),
            "bf16": hp.get("bf16"),
            "amp_dtype": hp.get("amp_dtype"),
            "grad_ckpt": hp.get("gradient_checkpointing"),
            "lora_r": lora.get("r"),
            "lora_targets": ",".join(lora.get("target_modules", [])) if isinstance(lora.get("target_modules"), list) else lora.get("target_modules"),
            "modules_to_save": ",".join(lora.get("modules_to_save", [])) if isinstance(lora.get("modules_to_save"), list) else lora.get("modules_to_save"),
            "final_acc": final.get("accuracy"),
            "final_rho": final.get("rho"),
            "best_epoch": best_epoch,
            "best_acc": best.get("accuracy"),
            "best_rho": best.get("rho"),
            "train_runtime_s": comp.get("total_training_runtime_seconds"),
            "mean_latency_s": comp.get("mean_inference_latency_seconds_per_example"),
            "metadata_file": str(path),
        })

if not rows:
    print("No DeBERTa LoRA fine-tuning metadata rows found yet.")
else:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)

        order = {
            "cross-encoder/nli-deberta-v3-xsmall": 0,
            "cross-encoder/nli-deberta-v3-small": 1,
            "cross-encoder/nli-deberta-v3-base": 2,
            "cross-encoder/nli-deberta-v3-large": 3,
            "sileod/deberta-v3-base-tasksource-nli": 4,
            "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli": 5,
        }
        df["_order"] = df["base_model_name"].map(order).fillna(99)
        display(df.sort_values(["_order", "parameter_count"]).drop(columns=["_order"]))
    except Exception:
        for row in rows:
            print(row)

backup_outputs_to_drive()

,base_model_name,parameter_count,trainable_params,trainable_frac,label_mode,split_mode,num_train,num_eval,epochs,lr,...,lora_targets,modules_to_save,final_acc,final_rho,best_epoch,best_acc,best_rho,train_runtime_s,mean_latency_s,metadata_file
4,cross-encoder/nli-deberta-v3-xsmall,71420930,590593,0.008269,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.79,0.674796,4,0.79,0.674796,159.471625,0.037172,/content/model_experiments_colab/outputs/metad...
3,cross-encoder/nli-deberta-v3-small,142929410,1033729,0.007232,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.77,0.657250,4,0.77,0.657250,86.165519,0.020199,/content/model_experiments_colab/outputs/metad...
1,cross-encoder/nli-deberta-v3-base,185899010,1476097,0.007940,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.76,0.657220,3,0.76,0.657220,162.050873,0.038530,/content/model_experiments_colab/outputs/metad...
2,cross-encoder/nli-deberta-v3-large,438472706,3409921,0.007777,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.76,0.645992,2,0.76,0.645992,308.938764,0.084098,/content/model_experiments_colab/outputs/metad...
5,sileod/deberta-v3-base-tasksource-nli,185899010,1476097,0.007940,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.81,0.716244,4,0.81,0.716244,166.022628,0.039560,/content/model_experiments_colab/outputs/metad...
0,MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-...,438472706,3409921,0.007777,soft,train_path_split_0.20,399,100,4,0.0002,...,"query_proj,key_proj,value_proj","classifier,pooler,classifier,score",0.77,0.685755,3,0.77,0.685755,311.097256,0.075733,/content/model_experiments_colab/outputs/metad...


Backed up outputs -> /content/drive/MyDrive/thesis_colab/outputs_deberta_lora_finetune_colab_vscode/20260510_111929/outputs
